In [34]:
import pennylane as qml
from pennylane import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import pandas as pd

# Load and preprocess
df = pd.read_csv("../Datasets For Classification/SuperStore/SampleSuperstore.csv", encoding='ISO-8859-1')
df = df[['Sales', 'Quantity', 'Discount', 'Profit']].dropna()
df['Target'] = df['Profit'].apply(lambda x: 1 if x > 0 else 0)

X = df[['Sales', 'Quantity', 'Discount']].values
y = df['Target'].values

X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Quantum circuit setup
n_qubits = 3
dev = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev, interface="autograd")
def circuit(weights, x):
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)
    qml.templates.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

# Variational classifier
def variational_classifier(weights, x):
    return circuit(weights, x)

# Loss function
def square_loss(weights, X, Y):
    loss = 0
    for x, y in zip(X, Y):
        pred = variational_classifier(weights, x)
        loss += (pred - y) ** 2
    return loss / len(X)

# Accuracy
def accuracy(weights, X, Y):
    preds = [1 if variational_classifier(weights, x) > 0 else 0 for x in X]
    return np.mean(preds == Y)

# Training
weights = 0.01 * np.random.randn(4, n_qubits, requires_grad=True)
opt = qml.optimize.AdamOptimizer(stepsize=0.1)

for it in range(25):
    weights, loss = opt.step_and_cost(lambda w: square_loss(w, X_train[:100], y_train[:100]), weights)
    if (it + 1) % 5 == 0:
        acc = accuracy(weights, X_test[:100], y_test[:100])
        print(f"Step {it+1} | Loss: {loss:.4f} | Accuracy: {acc:.3f}")

# Evaluation
preds = [1 if variational_classifier(weights, x) > 0 else 0 for x in X_test[:100]]
print("\nClassification Report:\n")
print(classification_report(y_test[:100], preds))


Step 5 | Loss: 0.3727 | Accuracy: 0.780
Step 10 | Loss: 0.3582 | Accuracy: 0.780
Step 15 | Loss: 0.3487 | Accuracy: 0.780
Step 20 | Loss: 0.3466 | Accuracy: 0.770
Step 25 | Loss: 0.3497 | Accuracy: 0.770

Classification Report:

              precision    recall  f1-score   support

           0       0.43      0.14      0.21        22
           1       0.80      0.95      0.87        78

    accuracy                           0.77       100
   macro avg       0.61      0.54      0.54       100
weighted avg       0.71      0.77      0.72       100

